    # Week 2 Completion Project

    To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
    and responds with an explanation. This is a tool that you will be able to use yourself during the course!


    Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.
    This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!
    If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

In [154]:
# imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

from IPython.display import Markdown, display

In [155]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

load_dotenv(override=True)
search_api_key = os.getenv('SEARCH_API_KEY')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [156]:
MODEL='gpt-oss:20b'
# response = ollama.chat.completions.create(model="gpt-oss:20b", messages=[{"role": "user", "content": "Tell me a fun fact"}])
# response.choices[0].message.content

In [157]:

system_message = """You are a Technical Explanation Assistant.

Your job is to take any technical question (programming, computer science, AI, networking, security, data engineering, etc.) and respond with a clear, structured, beginner-to-intermediate friendly explanation.

Follow these rules:

1. Start with a concise high-level overview (2-4 sentences).
2. Prefer practical intuition over abstract theory.
3. If relevant, include Real-world analogy
4. End with a short summary.

Tone:
- Professional, clear, and precise.
- Assume the user has basic technical literacy.
- Avoid unnecessary verbosity or academic language.

Formatting:
- Use headings and bullet points when appropriate.
- Keep paragraphs short.
- Highlight important terms in bold.

If the question is ambiguous, make a reasonable assumption and proceed.

Primary goal: maximize understanding."""

In [158]:
search_function = {
    "name": "get_docs_links",
    "description": "Search the web and return in-depth official documentation links for a technical topic provided by the user.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Technical topic or keywords to search for (e.g. 'FastAPI authentication docs', 'Pydantic BaseModel guide')."
            },
            "source_preference": {
                "type": "string",
                "description": "Optional preference like 'official', 'github', or 'tutorial'.",
                "enum": ["official", "github", "tutorial", "any"]
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": search_function}]

In [159]:
def get_docs_links(user_input):
    BASE_URL = "https://www.searchapi.io/api/v1/search"

    params = {
        "engine": "google",
        "q": f"{user_input}",
        "api_key": search_api_key,
    }

    response = requests.get(BASE_URL, params=params)

    response.raise_for_status()
    data = response.json()

    results = []
    for result in data.get("organic_results", []):
        results.append({
        "title": result["title"],
        "link": result["link"]
    })
        
    answer = []

    prompt = """
    You are a strict filter.

    You receive a list of objects like:
    {'title': '...', 'link': '...'}

    Your task:
    - Return ONLY links that belong to OFFICIAL documentation websites.
    - Ignore GitHub repos, Wikipedia, PyPI, blogs, tutorials, YouTube, etc.
    - Respond ONLY with valid JSON in this format:

    {
    "official_docs": ["https://example.com"]
    }
    """

    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": json.dumps(results)}
            ],
            response_format={"type": "json_object"}
        )

        data = response.choices[0].message.content
        parsed = json.loads(data)

        answer.extend(parsed.get("official_docs", []))

    except Exception as e:
        print("LLM parsing failed:", e)

    return answer
            
    

    

In [160]:
def chat(message, history):
    # 1. format history explicitly for safety/consistency
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # 2. Construct the message payload
    # Note: We will define system_message in the next section
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
       
        tool_call = message_obj.tool_calls[0]
        
        
        args = json.loads(tool_call.function.arguments)
        query = args["query"]
        
        required_link = get_docs_links(query)

        response = {
            "role": "tool",
            "content": F"{required_link}",
            "tool_call_id": tool_call.id
        }

    
        messages.append(message_obj)
        messages.append(response)
        
        response = client.chat.completions.create(model=MODEL, messages=messages)
        
    return response.choices[0].message.content     



In [161]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    theme=gr.themes.Monochrome()
).launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.
